# Synthetic cell-mixture study

This notebook measures Xenocomm across controlled human–mouse mixtures for the melanoma PDX and 10x HGMM datasets. Each mixture contains an explicit set of human and mouse cells. Ligand contribution is computed cell by cell from raw counts, then summed separately by species before fitting; no expression profiles are interpolated.

Counts are standardized to a common library depth within each selected cell and converted to detection signal. Human and mouse signal are aggregated over the complete mixed pool, then split into additive complementary-log-log contributions. A 50% human cell mixture therefore reflects the ligand output of the sampled cells rather than being forced to 50% human ligand expression.

PDX expression is rebuilt from raw counts in `.raw.X`: normalize each cell to 10,000 counts, then apply natural-log1p. The original counts are retained in `layers["counts"]`; saved expression and gene-selection statistics are not used.


In [ ]:
from pathlib import Path
from shutil import copyfileobj
from urllib.request import Request, urlopen

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy import sparse
from scipy.stats import spearmanr
import xenocomm as xc

sns.set_theme(context="notebook", style="whitegrid")

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "data").is_dir():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"
PDX_DIR = NOTEBOOK_DIR / "data/melanoma_pdx_10k"
TENX_DIR = NOTEBOOK_DIR / "data/10x_hgmm"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs/mixtures_10k"
TENX_H5 = TENX_DIR / "10k_hgmm_3p_gemx_count_sample_filtered_feature_bc_matrix.h5"
TENX_URL = (
    "https://cf.10xgenomics.com/samples/cell-exp/8.0.0/"
    "10k_hgmm_3p_gemx_10k_hgmm_3p_gemx/"
    "10k_hgmm_3p_gemx_10k_hgmm_3p_gemx_count_sample_filtered_feature_bc_matrix.h5"
)

TENX_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not TENX_H5.exists():
    download = TENX_H5.with_suffix(".download")
    request = Request(TENX_URL, headers={"Range": "bytes=0-", "User-Agent": "Mozilla/5.0"})
    with urlopen(request) as source, download.open("wb") as destination:
        copyfileobj(source, destination)
    download.replace(TENX_H5)

def read_pdx_counts(path):
    stored = ad.read_h5ad(path)
    if stored.raw is None:
        raise ValueError(f"{path} must contain raw counts in .raw.X")
    counts = sparse.csr_matrix(stored.raw.X, copy=True)
    if (
        not np.isfinite(counts.data).all()
        or np.any(counts.data < 0)
        or np.any(counts.data != np.floor(counts.data))
    ):
        raise ValueError(f"{path}: .raw.X must contain nonnegative integer counts")
    if np.any(np.asarray(counts.sum(axis=1)).ravel() <= 0):
        raise ValueError(f"{path}: every cell must have a positive count total")
    data = ad.AnnData(
        X=counts.astype(np.float32),
        obs=stored.obs[["sample", "cell_type"]].copy(),
        var=stored.raw.var[["gene_id"]].copy(),
    )
    data.layers["counts"] = counts
    sc.pp.normalize_total(data, target_sum=10_000)
    sc.pp.log1p(data)
    return data


pdx_mouse = read_pdx_counts(PDX_DIR / "adata_mouse.h5ad")
pdx_human = read_pdx_counts(PDX_DIR / "adata_human.h5ad")
tenx_raw = sc.read_10x_h5(TENX_H5)


In [ ]:
human_genes = tenx_raw.var["genome"].eq("GRCh38").to_numpy()
mouse_genes = tenx_raw.var["genome"].eq("GRCm39").to_numpy()
human_umis = np.asarray(tenx_raw[:, human_genes].X.sum(axis=1)).ravel()
mouse_umis = np.asarray(tenx_raw[:, mouse_genes].X.sum(axis=1)).ravel()
total_umis = human_umis + mouse_umis
human_cells = human_umis / total_umis >= 0.9
mouse_cells = mouse_umis / total_umis >= 0.9


def collapse_species(cell_mask, gene_mask, prefix):
    subset = tenx_raw[cell_mask, gene_mask]
    symbols = pd.Index(subset.var_names.str.removeprefix(prefix), name="gene_symbol")
    gene_ids = subset.var["gene_ids"].astype(str).str.removeprefix(prefix)
    unique_symbols = symbols.drop_duplicates()
    output_index = pd.Series(np.arange(len(unique_symbols)), index=unique_symbols)
    columns = output_index.loc[symbols].to_numpy()
    projection = sparse.csr_matrix(
        (np.ones(len(symbols), dtype=np.int32), (np.arange(len(symbols)), columns)),
        shape=(len(symbols), len(unique_symbols)),
    )
    counts = (subset.X.tocsr().astype(np.int32) @ projection).tocsr()
    ids = pd.Series(gene_ids.to_numpy(), index=symbols).groupby(level=0, sort=False).agg(";".join)
    var = pd.DataFrame({"gene_ids": ids.loc[unique_symbols].to_numpy()}, index=unique_symbols)
    return counts, var, subset.obs_names.copy()


collapsed = {
    "human": collapse_species(human_cells, human_genes, "GRCh38_"),
    "mouse": collapse_species(mouse_cells, mouse_genes, "GRCm39_"),
}
shared_genes = sorted(
    {gene.casefold() for gene in collapsed["human"][1].index}
    & {gene.casefold() for gene in collapsed["mouse"][1].index}
)


def normalize_species(species, cell_line):
    counts, var, obs_names = collapsed[species]
    lookup = {gene.casefold(): index for index, gene in enumerate(var.index)}
    shared_indices = np.asarray([lookup[gene] for gene in shared_genes])
    library_size = np.asarray(counts[:, shared_indices].sum(axis=1)).ravel()
    expression = counts.astype(np.float32).multiply((10_000 / library_size)[:, None]).tocsr()
    np.log1p(expression.data, out=expression.data)
    obs = pd.DataFrame({"cell_line": cell_line, "species": species}, index=obs_names)
    result = ad.AnnData(expression, obs=obs, var=var)
    result.layers["counts"] = counts
    if species == "mouse":
        dispersion_input = ad.AnnData(counts[:, shared_indices].copy(), var=var.iloc[shared_indices].copy())
        sc.pp.log1p(dispersion_input)
        sc.pp.highly_variable_genes(dispersion_input)
        dispersions = pd.Series(
            dispersion_input.var["dispersions_norm"].to_numpy(),
            index=dispersion_input.var_names.str.casefold(),
        )
        result.var["dispersions_norm"] = [dispersions.get(gene.casefold(), np.nan) for gene in var.index]
    return result


tenx_human = normalize_species("human", "HEK293T")
tenx_mouse = normalize_species("mouse", "NIH3T3")
print(f"PDX: {pdx_human.n_obs:,} human + {pdx_mouse.n_obs:,} mouse cells")
print(f"10x: {tenx_human.n_obs:,} human + {tenx_mouse.n_obs:,} mouse cells")


In [ ]:
RATES = (0.0, 0.05, 0.1, 0.25, 0.5, 1.0)
TRIALS = 5
POOL_CELLS = 5_000
POSTERIOR_SAMPLES = 2_000
LIGAND_REFERENCE_DEPTH = 25_000.0
LIGAND_DEGREE_POWER = 0.25


def stratified_validation_indices(obs, size, seed):
    strata = obs[["sample", "cell_type"]].astype(str).agg("|".join, axis=1)
    counts = strata.value_counts().sort_index()
    exact = counts.to_numpy() * size / len(obs)
    quotas = np.floor(exact).astype(int)
    quotas[np.argsort(-(exact - quotas), kind="stable")[: size - quotas.sum()]] += 1
    rng = np.random.RandomState(seed)
    held_out = np.concatenate([
        rng.choice(np.flatnonzero(strata.to_numpy() == label), quota, replace=False)
        for label, quota in zip(counts.index, quotas, strict=True)
    ])
    rng.shuffle(held_out)
    return np.setdiff1d(np.arange(len(obs)), held_out), held_out


def random_validation_indices(size_total, size, seed):
    rng = np.random.RandomState(seed)
    held_out = rng.choice(size_total, size, replace=False)
    rng.shuffle(held_out)
    return np.setdiff1d(np.arange(size_total), held_out), held_out


def raw_counts_and_names(adata):
    if adata.raw is not None:
        return sparse.csr_matrix(adata.raw.X), adata.raw.var_names.astype(str)
    return sparse.csr_matrix(adata.layers["counts"]), adata.var_names.astype(str)


def ligand_signal_matrix(adata, genes):
    counts, names = raw_counts_and_names(adata)
    lookup = {gene.casefold(): index for index, gene in enumerate(names)}
    columns = [lookup[gene.casefold()] for gene in genes]
    depth = np.maximum(np.asarray(counts.sum(axis=1)).ravel(), 1)
    signal = counts[:, columns].tocoo().astype(np.float64)
    signal.data = 1 - np.exp(
        -signal.data * LIGAND_REFERENCE_DEPTH / depth[signal.row]
    )
    return signal.tocsr()


def mixed_ligand_contribution(mouse_signal, human_signal, mouse_indices, human_indices, lr_matrix):
    total_cells = len(mouse_indices) + len(human_indices)
    mouse_prevalence = np.asarray(mouse_signal[mouse_indices].sum(axis=0)).ravel() / total_cells
    human_prevalence = np.asarray(human_signal[human_indices].sum(axis=0)).ravel() / total_cells
    prevalence = np.stack([mouse_prevalence, human_prevalence])
    total_prevalence = prevalence.sum(axis=0)
    total_rate = -np.log1p(-np.clip(total_prevalence, 0, 1 - 1e-8))
    abundance = np.divide(
        prevalence,
        total_prevalence,
        out=np.zeros_like(prevalence),
        where=total_prevalence > 0,
    ) * total_rate
    degree = np.maximum(np.asarray(lr_matrix).sum(axis=1), 1.0)
    return (abundance / degree[None, :] ** LIGAND_DEGREE_POWER).astype(np.float32)


In [ ]:
pdx_training, pdx_validation = stratified_validation_indices(pdx_mouse.obs, 4096, 20260717)
tenx_training, tenx_validation = random_validation_indices(tenx_mouse.n_obs, 687, 20260718)
studies = {
    "PDX": {
        "mouse": pdx_mouse, "human": pdx_human,
        "training": pdx_training, "validation": pdx_validation,
        "batch_size": 1024, "steps_per_batch": 250, "epochs": 5,
        "training_seed": 20260716, "validation_seed": 20260718, "mixture_seed": 2026090100,
    },
    "10x": {
        "mouse": tenx_mouse, "human": tenx_human,
        "training": tenx_training, "validation": tenx_validation,
        "batch_size": 687, "steps_per_batch": 250, "epochs": 29,
        "training_seed": 20260717, "validation_seed": 20260719, "mixture_seed": 2026090200,
    },
}

for name, study in studies.items():
    study["network"] = xc.prepare_network(study["mouse"], study["human"], dispersion_cutoff=-10)
    study["mouse_signal"] = ligand_signal_matrix(study["mouse"], study["network"]["ligands"])
    study["human_signal"] = ligand_signal_matrix(study["human"], study["network"]["human_ligands"])
    if POOL_CELLS > min(study["mouse"].n_obs, study["human"].n_obs):
        raise ValueError(f"{name} does not have {POOL_CELLS:,} cells per species")
    print(
        f"{name}: {len(study['network']['ligands']):,} ligands, "
        f"{len(study['network']['receptors']):,} receptors, "
        f"{len(study['network']['targets']):,} targets"
    )


In [ ]:
ligand_runs = []
receptor_runs = []
mixture_inputs = []
for dataset, study in studies.items():
    network = study["network"]
    dataset_key = dataset.lower()
    for trial in range(TRIALS):
        mouse_stream, human_stream = np.random.SeedSequence(study["mixture_seed"] + trial).spawn(2)
        mouse_order = np.random.default_rng(mouse_stream).permutation(study["mouse"].n_obs)
        human_order = np.random.default_rng(human_stream).permutation(study["human"].n_obs)
        for rate in RATES:
            human_count = round(POOL_CELLS * rate)
            mouse_count = POOL_CELLS - human_count
            human_indices = human_order[:human_count]
            mouse_indices = mouse_order[:mouse_count]
            rate_key = f"{rate:g}".replace(".", "p")
            stem = f"{dataset_key}_rate_{rate_key}_trial_{trial + 1:02d}"
            ligand_path = OUTPUT_DIR / f"{stem}_ligands.parquet"
            receptor_path = OUTPUT_DIR / f"{stem}_receptors.parquet"
            input_path = OUTPUT_DIR / f"{stem}_input.parquet"
            if ligand_path.exists() and receptor_path.exists() and input_path.exists():
                ligand_table = pd.read_parquet(ligand_path)
                receptor_table = pd.read_parquet(receptor_path)
                input_table = pd.read_parquet(input_path)
            else:
                print(f"Training {dataset}, human cell fraction {rate:g}, trial {trial + 1}")
                abundance = mixed_ligand_contribution(
                    study["mouse_signal"],
                    study["human_signal"],
                    mouse_indices,
                    human_indices,
                    network["ligand_receptor_matrix"],
                )
                total = abundance.sum(axis=0)
                input_table = pd.DataFrame({
                    "ligand": network["ligands"],
                    "mouse_contribution": abundance[0],
                    "human_contribution": abundance[1],
                    "actual_human_fraction": np.divide(
                        abundance[1], total, out=np.zeros_like(total), where=total > 0
                    ),
                })
                model = xc.XenocommModel(
                    study["mouse"][study["training"]],
                    **network,
                    mean_ligand=abundance,
                    receptor_target_mode="learned",
                    training_seed=study["training_seed"] + 2 * trial,
                    posterior_seed=study["training_seed"] + 2 * trial + 1,
                    batch_size=study["batch_size"],
                    steps_per_batch=study["steps_per_batch"],
                    epochs=study["epochs"],
                )
                model.train(
                    validation_mouse=study["mouse"][study["validation"]],
                    absolute_tolerance=0.001 * study["batch_size"],
                    patience=3,
                    min_evaluations=5,
                    validation_seed=study["validation_seed"],
                )
                samples = model.sample(POSTERIOR_SAMPLES)
                parameters = model.get_parameters()
                ligand_table = xc.ligand_result_table(
                    model.ligands, samples, model.mean_ligand_np,
                    model.ligand_receptor_matrix_np, xc.get_receptor_sensitivity(parameters),
                )
                receptor_table = xc.receptor_marginal_df(model, samples, parameters)
                receptor_table["total_activation"] = receptor_table["mouse"] + receptor_table["delta"]
                for frame in (ligand_table, receptor_table, input_table):
                    frame["dataset"] = dataset
                    frame["human_cell_fraction"] = rate
                    frame["trial"] = trial + 1
                    frame["human_cells"] = human_count
                    frame["mouse_cells"] = mouse_count
                ligand_table.to_parquet(ligand_path, index=False)
                receptor_table.to_parquet(receptor_path, index=False)
                input_table.to_parquet(input_path, index=False)
            ligand_runs.append(ligand_table)
            receptor_runs.append(receptor_table)
            mixture_inputs.append(input_table)


In [ ]:
ligand_results = pd.concat(ligand_runs, ignore_index=True)
receptor_results = pd.concat(receptor_runs, ignore_index=True)
input_results = pd.concat(mixture_inputs, ignore_index=True)
ligand_results.to_parquet(OUTPUT_DIR / "ligand_results.parquet", index=False)
receptor_results.to_parquet(OUTPUT_DIR / "receptor_results.parquet", index=False)
input_results.to_parquet(OUTPUT_DIR / "mixture_inputs.parquet", index=False)

summary_rows = []
for (dataset, rate, trial), table in ligand_results.groupby(
    ["dataset", "human_cell_fraction", "trial"], sort=True
):
    inputs = input_results.query(
        "dataset == @dataset and human_cell_fraction == @rate and trial == @trial"
    ).set_index("ligand").loc[table["ligand"]]
    contribution_total = inputs["human_contribution"].sum() + inputs["mouse_contribution"].sum()
    delta_total = table["delta_h_mean"].sum() + table["delta_mouse_mean"].sum()
    correlation = np.nan
    if 0 < rate < 1:
        correlation = spearmanr(inputs["actual_human_fraction"], table["human_fraction_mean"]).statistic
    called = table[table["called"]]
    summary_rows.append({
        "dataset": dataset, "human_cell_fraction": rate, "trial": trial,
        "human_cells": int(inputs["human_cells"].iloc[0]),
        "mouse_cells": int(inputs["mouse_cells"].iloc[0]),
        "actual_human_ligand_fraction": inputs["human_contribution"].sum() / contribution_total,
        "inferred_human_fraction": table["delta_h_mean"].sum() / delta_total if delta_total > 0 else 0.0,
        "ligand_fraction_spearman": correlation,
        "detected_ligands": int(table["called"].sum()),
        "total_detected_human_contribution": called["delta_h_mean"].sum(),
    })
mixture_summary = pd.DataFrame(summary_rows)
mixture_summary.to_parquet(OUTPUT_DIR / "mixture_summary.parquet", index=False)
mixture_summary


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.lineplot(
    data=mixture_summary, x="human_cell_fraction", y="actual_human_ligand_fraction",
    hue="dataset", marker="o", errorbar="sd", ax=axes[0],
)
axes[0].plot([0, 1], [0, 1], color="black", linestyle="--", linewidth=1)
axes[0].set(xlabel="Human cell fraction", ylabel="Human ligand contribution", title="Contribution from sampled cells")
sns.lineplot(
    data=mixture_summary, x="human_cell_fraction", y="total_detected_human_contribution",
    hue="dataset", marker="o", errorbar="sd", ax=axes[1], legend=False,
)
axes[1].set(xlabel="Human cell fraction", ylabel="Total human counterfactual activation", title="Recovered human signal")
sns.lineplot(
    data=mixture_summary, x="human_cell_fraction", y="detected_ligands",
    hue="dataset", marker="o", errorbar="sd", ax=axes[2], legend=False,
)
axes[2].set(xlabel="Human cell fraction", ylabel="Detected ligands", title="Ligand detection")
plt.tight_layout()


In [ ]:
middle = ligand_results.query("human_cell_fraction == 0.5").merge(
    input_results.query("human_cell_fraction == 0.5")[
        ["dataset", "trial", "ligand", "actual_human_fraction"]
    ],
    on=["dataset", "trial", "ligand"],
)
fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(
    data=middle, x="actual_human_fraction", y="human_fraction_mean",
    hue="dataset", alpha=0.35, s=18, ax=ax,
)
ax.plot([0, 1], [0, 1], color="black", linestyle="--", linewidth=1)
ax.set(
    xlabel="Human fraction of sampled ligand contribution",
    ylabel="Inferred human signaling fraction",
    title="Per-ligand recovery at 50% human cells",
)
plt.tight_layout()
